# DS2002 · Environment and GitHub Setup

**Lab — 2026-08-28 · Fall 2026**  

---

## Lab 01 — Environment and GitHub Setup

Two things have to be true before next week: you can run Python in a browser notebook, and you own a GitHub repository with your work in it. This lab proves both.

The Git section is the longest part on purpose. Git is the one tool in this course that punishes you later if you fake your way through it now, so we are going to run real commands and read real output before you touch your own repository.

**Submit in Canvas:** your repository URL and the link to this notebook.

### Part 1 — Where is this notebook actually running?

Kaggle and Colab look similar and put files in different places. Knowing which one you are in saves you an hour later when a file path does not resolve.

**TODO:** set `name`, then run the cell.

This code was mostly given to me, I just changed the `name` variable to my own
name. The code checks whether I'm running in Kaggle, Google Colab, or something
else, and prints that along with my Python version and
current working directory

In [1]:
import sys, os, platform

name = 'Malia Solomon'

if os.path.exists('/kaggle'):
    where = 'Kaggle'
elif 'google.colab' in sys.modules:
    where = 'Colab'
else:
    where = 'something else (local Jupyter?)'

print(f'Hello, {name}.')
print('Environment:', where)
print('Python:', platform.python_version())
print('Working directory:', os.getcwd())

Hello, Malia Solomon.
Environment: Kaggle
Python: 3.13.15
Working directory: /content


Findings: The output showed my environment as `Kaggle`, which was surprising since I was running the notebook in Colab.  After digging into it, I found that Colab has an empty `/kaggle` mount point by default. Since the code
checks `/kaggle` first before checking for Colab, it triggered the wrong branch. Furthermore, the output showed that I was working withPython `3.13.15` and shows my current working directory is `/content`.

### Part 2 — Python warm-up

**TODO:** build a list of five numbers, then print the total and the average. Write the loop yourself for the total instead of calling `sum()` — later in the course you will be reading loops that someone else wrote, and this is the shape they take.

For this question, I first made a list called `numbers` with five values in it (1 through 5), then set up a variable called `total` starting at 0. I used a for loop to go through each number in the list one at a time and add it to `total`. After the loop was done adding everything up, I found the average by dividing the established total by how many numbers were in the list. I used `len(numbers)` instead of just typing 5, so it would still work even if I added more numbers to the list later.

In [2]:
numbers = [1, 2, 3, 4, 5]

total = 0
for n in numbers:
    total += n

average = total/len(numbers)

print('total:', total)
print('average:', average)

total: 15
average: 3.0


Findings:
The list `[1, 2, 3, 4, 5]` has a total sum of 15, and dividing that by the
5 numbers in the list gives an average of 3.0.

### Part 3 — A small table

**TODO:** build a DataFrame of three tailgate items with `item` and `price` columns, then print the row with the highest price.

For this question I used pandas to make a DataFrame called `menu` with two columns — `item` and `price` — for a burger, fries, and a soda. To find the most expensive item, I used `.idxmax()` on the `price` column, which
gives back the row number of the largest value instead of only the value itself. Then I used `.loc[]` with that index to pull out the whole row, so I could see both the item name as well as its price.

In [3]:
import pandas as pd

menu = pd.DataFrame({
    'item' : ['burger', 'fries', 'soda'],
    'price': [10, 5, 2]
})

print(menu.loc[menu['price'].idxmax()])

item     burger
price        10
Name: 0, dtype: object


Findings: Running the code shows that the `burger` is the most expensive item on the menu at $10. This makes sense just by eyeballing the prices (10, 5, and 2), the burger is clearly the highest.

---

## Part 4 — What Git actually is

Git is a tool for recording the history of a folder. That is the whole idea. Everything else is vocabulary for that one job.

| Word | What it means |
|---|---|
| **repository** (repo) | A folder Git is tracking, plus its whole history |
| **commit** | One saved snapshot of the folder, with a message saying what changed |
| **staging area** | The list of changes you have selected to go in the *next* commit |
| **remote** | A copy of the repo somewhere else. Yours will live on GitHub |
| **origin** | The default nickname for your remote |
| **push / pull** | Send your commits to the remote / bring the remote's commits down |

The loop you will run every week:

```bash
git add <files>          # choose what goes in the snapshot
git commit -m "message"   # take the snapshot
git push                 # send it to GitHub
```

GitHub is not Git. Git is the program that records history; GitHub is a website that hosts a copy of it and adds pull requests, issues, and permissions on top.

### Part 5 — Run real Git, right here

The next several cells build a throwaway repository in a temporary folder and run actual Git commands against it. Nothing here touches your own work — it exists so you can read Git's real output before it matters.

Run them in order and read what comes back.

In [4]:
import subprocess, tempfile, os

SANDBOX = tempfile.mkdtemp(prefix='ds2002-git-')

def git(cmd, cwd=SANDBOX):
    """Run a command in the sandbox folder and print exactly what git says back."""
    done = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    print('$', cmd)
    out = (done.stdout + done.stderr).strip()
    print(out if out else '(no output)')
    print()
    return done

print('sandbox:', SANDBOX)
git('git --version')

sandbox: /tmp/ds2002-git-3p6weth5
$ git --version
git version 2.34.1



CompletedProcess(args='git --version', returncode=0, stdout='git version 2.34.1\n', stderr='')

`git init` creates the repository. Git also refuses to commit until it knows who you are, so we set a name and email for this sandbox only.

In [5]:
git('git init -b main')
git("git config user.name 'DS2002 Student'")
git("git config user.email 'student@example.com'")

$ git init -b main
Initialized empty Git repository in /tmp/ds2002-git-3p6weth5/.git/

$ git config user.name 'DS2002 Student'
(no output)

$ git config user.email 'student@example.com'
(no output)



CompletedProcess(args="git config user.email 'student@example.com'", returncode=0, stdout='', stderr='')

Now create a file and ask Git what it sees. `??` means *untracked* — the file exists, but Git has been told nothing about it.

In [6]:
with open(os.path.join(SANDBOX, 'notes.md'), 'w') as f:
    f.write('# Game day notes\n\nPonchos sell when it rains.\n')

git('git status --short')

$ git status --short
?? notes.md



CompletedProcess(args='git status --short', returncode=0, stdout='?? notes.md\n', stderr='')

`git add` moves the file into the staging area. Watch the status letter change from `??` to `A` — Git now plans to include it in the next commit.

In [7]:
git('git add notes.md')
git('git status --short')

$ git add notes.md
(no output)

$ git status --short
A  notes.md



CompletedProcess(args='git status --short', returncode=0, stdout='A  notes.md\n', stderr='')

`git commit` takes the snapshot. `git log --oneline` lists the history: one short hash per commit, plus its message. That hash is how you refer to a specific snapshot forever.

In [8]:
git("git commit -m 'Add game day notes'")
git('git log --oneline')

$ git commit -m 'Add game day notes'
[main (root-commit) dc58bd2] Add game day notes
 1 file changed, 3 insertions(+)
 create mode 100644 notes.md

$ git log --oneline
dc58bd2 Add game day notes



CompletedProcess(args='git log --oneline', returncode=0, stdout='dc58bd2 Add game day notes\n', stderr='')

Now change the file without committing, and ask for the difference. Lines starting with `+` are additions, `-` are removals. This is the single most useful Git command for answering "what am I about to commit?"

In [9]:
with open(os.path.join(SANDBOX, 'notes.md'), 'a') as f:
    f.write('\nRefunds come through as negative units.\n')

git('git diff')

$ git diff
diff --git a/notes.md b/notes.md
index 8e3a320..2caadbe 100644
--- a/notes.md
+++ b/notes.md
@@ -1,3 +1,5 @@
 # Game day notes
 
 Ponchos sell when it rains.
+
+Refunds come through as negative units.



CompletedProcess(args='git diff', returncode=0, stdout='diff --git a/notes.md b/notes.md\nindex 8e3a320..2caadbe 100644\n--- a/notes.md\n+++ b/notes.md\n@@ -1,3 +1,5 @@\n # Game day notes\n \n Ponchos sell when it rains.\n+\n+Refunds come through as negative units.\n', stderr='')

In [10]:
git('git add -A')
git("git commit -m 'Note how refunds are recorded'")
git('git log --oneline')
git('git show --stat HEAD')

$ git add -A
(no output)

$ git commit -m 'Note how refunds are recorded'
[main c524d5c] Note how refunds are recorded
 1 file changed, 2 insertions(+)

$ git log --oneline
c524d5c Note how refunds are recorded
dc58bd2 Add game day notes

$ git show --stat HEAD
commit c524d5ca63b19b8aa730856b9b60f206315783d9
Author: DS2002 Student <student@example.com>
Date:   Mon Aug 31 00:42:42 2026 +0000

    Note how refunds are recorded

 notes.md | 2 ++
 1 file changed, 2 insertions(+)



CompletedProcess(args='git show --stat HEAD', returncode=0, stdout='commit c524d5ca63b19b8aa730856b9b60f206315783d9\nAuthor: DS2002 Student <student@example.com>\nDate:   Mon Aug 31 00:42:42 2026 +0000\n\n    Note how refunds are recorded\n\n notes.md | 2 ++\n 1 file changed, 2 insertions(+)\n', stderr='')

Two commits, and the file's whole history is recoverable. `HEAD` always means "the commit I am currently sitting on."

**TODO:** add a third commit to the sandbox. Create a file called `vendors.md` with any content, stage it, commit it with a message that says what you did, and print the log.

For this question, I created a new file called `vendors.md`. I then used `open()` with `'w'` mode to create the file and write that text into it, using `os.path.join()` to make sure it saves in the right folder (`SANDBOX`). Once the file was created, I used git to add it to version control. First I ran
`git add vendors.md` to move the file into the staging area, then `git commit -m 'Add vendors file'`to save that change with a message describing what I did. Finally, I ran `git log --oneline` so I could see the commit history and confirm my new commit actually went through.

In [11]:
with open(os.path.join(SANDBOX, 'vendors.md'), 'w') as f:
    f.write('# Vendors\n\nLocal food vendors.\n')

git('git add vendors.md')

git("git commit -m 'Add vendors file'")

git('git log --oneline')

$ git add vendors.md
(no output)

$ git commit -m 'Add vendors file'
[main 3c7f50d] Add vendors file
 1 file changed, 3 insertions(+)
 create mode 100644 vendors.md

$ git log --oneline
3c7f50d Add vendors file
c524d5c Note how refunds are recorded
dc58bd2 Add game day notes



CompletedProcess(args='git log --oneline', returncode=0, stdout='3c7f50d Add vendors file\nc524d5c Note how refunds are recorded\ndc58bd2 Add game day notes\n', stderr='')

Findings: After running this, the `vendors.md` file was created and successfully committed to git. I can see it show up in the `git log --oneline` output as the most recent commit with the message "Add vendors file". This confirms the file was saved and tracked correctly.

---

## Part 6 — Your real repository

Do this part outside the notebook, then come back and record the results.

1. Create a free account on <https://github.com> if you do not have one.
2. Create a new repository named `ds2002-fa26`. Check **Add a README file** so the repo starts with one commit in it.
3. Get this notebook into that repo, in a folder called `notebooks/01-foundations/`. Two ways to do that, and either is fine this week:

   **The web upload path.** Download this notebook (`File -> Download`), then on your repo page use **Add file -> Upload files**, drag it in, type a commit message, and commit. GitHub does the `add`/`commit` for you.

   **The command line path** — the one worth learning:

```bash
git clone https://github.com/<you>/ds2002-fa26.git
cd ds2002-fa26
mkdir -p notebooks/01-foundations
# move your downloaded .ipynb into that folder, then:
git add notebooks/01-foundations
git commit -m "Lab 01: environment and GitHub setup"
git push
```

4. Reload your repo page on GitHub and confirm the notebook is there.

**Naming.** Keep the course convention so files sort by date and read clearly:

`YYYY-MM-DD — Topic — Type.ipynb`, where Type is Lecture, Studio, Lab, or Template.

I used the web upload path to put my notebook into my repository. I downloaded my Colab notebook, and then uploaded the file into Github with a commit message and then committed the file. Then, I moved the file into the correct folder (`notebooks/01-foundations/`) with a second commit

I filled in the return piece of this function using an f-string to combine the
three pieces (date, topic, and kind of assignment) together with a " — " in between each one, and added `.ipynb` at the end since that's the file extension for notebooks. I also removed the `pass` placeholder since it's no longer needed once the function actually returns something

In [12]:
def make_filename(date, topic, kind):
    """Return the course-standard notebook filename."""
    return f'{date} — {topic} — {kind}.ipynb'

print(make_filename('2026-08-28', 'Environment and GitHub Setup', 'Lab'))
# should print: 2026-08-28 — Environment and GitHub Setup — Lab.ipynb

2026-08-28 — Environment and GitHub Setup — Lab.ipynb


Findings: Running the function with the test values gives back the filename
`"2026-08-28 — Environment and GitHub Setup — Lab.ipynb"`, which matches the
expected output. This confirms the accuracy of the function.

### Part 7 — What does not belong in a repository

A repo is for code and small text files. Three things cause real damage:

- **Secrets.** An API key in a commit is public the moment you push, and deleting it later does not remove it from the history. Rotate the key instead.
- **Big data.** GitHub rejects files over 100 MB, and a 90 MB CSV makes every clone slow forever. Commit the code that fetches or generates the data.
- **Junk.** Checkpoint folders, caches, and OS files add noise to every diff.

A `.gitignore` file tells Git to skip these. Run the cell, then add two patterns of your own — think about what your machine or environment leaves lying around.

I chose to exclude `*.log` because log files are just records of what happened when the code ran. They're generated automatically every time, they can get pretty large, and they don't add any value to the actual project. I also excluded `*.tmp` for a similar reason. Temporary files are meant to be short-lived and get created and deleted constantly while a program runs. Committing them would  clutter the repo with files that are not important. I used `*.log` and `*.tmp` instead of just `.log` and `.tmp` because the `*` is a
wildcard that matches any filename ending in that extension, allowing for all these file types to be ignored rather than only ones with exact matching naming.

In [13]:
gitignore = '''.ipynb_checkpoints/
__pycache__/
.DS_Store
.env
data/full/
*.log
*.tmp
'''

with open(os.path.join(SANDBOX, '.gitignore'), 'w') as f:
    f.write(gitignore)

print(gitignore)
git('git status --short')

.ipynb_checkpoints/
__pycache__/
.DS_Store
.env
data/full/
*.log
*.tmp

$ git status --short
?? .gitignore



CompletedProcess(args='git status --short', returncode=0, stdout='?? .gitignore\n', stderr='')

Findings: After writing the `.gitignore` file and running `git status --short`, none of the ignored file types (like `.log` or `.tmp` files) would show up as untracked files

### Part 8 — Notebooks in Git, honestly

A `.ipynb` file is JSON containing your code, your outputs, and an execution counter for every cell. That has consequences you should know now rather than discover during the midterm:

- **Diffs look terrible.** Changing one line of code can produce a hundred lines of diff because outputs and counters moved. That is normal. You are not expected to read notebook diffs line by line.
- **Restart and Run All before you commit.** Otherwise you commit outputs that do not match the code, which is worse than committing no outputs at all.
- **Do not have two people edit the same notebook at once.** The merge conflict lands in the middle of JSON and is miserable to resolve. On projects, split the work into separate files, or agree that one person owns the notebook at a time.

For this course, commit the notebook **with** its outputs. Graders need to see that your code ran.

### Part 9 — When Git pushes back

You will hit these. Recognizing the message is most of the fix.

| What you see | What happened | What to do |
|---|---|---|
| `Authentication failed` | GitHub stopped accepting account passwords in 2021 | Use a personal access token as the password, or set up SSH keys |
| `Updates were rejected... non-fast-forward` | The remote has commits you do not | `git pull` first, then push again |
| `nothing to commit, working tree clean` | You never saved the file, or it is ignored | Save it; check `.gitignore` |
| `file is 142.00 MB; this exceeds GitHub's limit` | A large data file got committed | Remove it, add it to `.gitignore`, commit the loader code instead |
| `Please tell me who you are` | No name/email configured | `git config --global user.name` and `user.email` |

When something looks wrong, run `git status` before you run anything else. It tells you which of the four places your changes are sitting in.

### Part 10 — Prove it

**TODO:** fill in the three values below from your real repository. The commit hash is the short code next to your commit on GitHub, or the first column of `git log --oneline`. The cell checks the shape of what you entered — it cannot check that the repo is really yours, so make sure it is.

For this question, I filled in my actual repo URL, commit hash, and notebook path, then ran a set of checks to make sure everything was formatted correctly before submitting. The checks confirm that my repo URL actually points to GitHub, that my commit hash looks like a real hash, and that my notebook path starts in the `notebooks/` folder and ends with `.ipynb`.

In [14]:
import re

my_repo_url = 'https://github.com/maliasolomon/ds2002-fa26'
my_commit_sha = '5c61885'
notebook_path_in_repo = 'notebooks/01-foundations/2026-08-28 — Environment and GitHub Setup — Lab.ipynb'

checks = {
    'repo URL points at github.com':
        my_repo_url.startswith('https://github.com/')
        and my_repo_url.count('/') >= 4,
    'commit hash looks like a hash':
        bool(re.fullmatch(r'[0-9a-f]{7,40}', my_commit_sha.strip())),
    'path is a notebook inside notebooks/':
        notebook_path_in_repo.startswith('notebooks/')
        and notebook_path_in_repo.endswith('.ipynb'),
}

for label, passed in checks.items():
    print('PASS' if passed else 'FAIL', '-', label)

print()
print('Ready to submit:', all(checks.values()))

PASS - repo URL points at github.com
PASS - commit hash looks like a hash
PASS - path is a notebook inside notebooks/

Ready to submit: True


Findings: All three checks came back as `PASS`, and the final output confirmed
`"Ready to submit: True"`. This indicates that my repo URL, commit hash, and notebook path are all correctly formatted, so I'm good for submission.

### Part 11 — Submit

Before you submit, from a fresh kernel: **Restart and Run All.** If any cell errors, fix it — a notebook that does not run is graded as it is.

Then in Canvas, hand in your repository URL and the link to this notebook.

### Stretch

Write `describe_number(n)` that reports whether a number is even or odd and whether it is negative, and run it over your list from Part 2. Commit it as a second commit with its own message — practice making commits that each do one thing.

For this stretch question, I wrote a function called `describe_number()` that uses `n % 2` to check for evenness, if there's no remainder when dividing by 2, the number is even, otherwise it's odd. Then I used a comparison (`n < 0`) to check the sign. Then I looped through the `numbers` list created earlier and called `describe_number()` on each one, so it prints a description for every number in the list

In [15]:
# stretch
def describe_number(n):
    if n % 2 == 0:
        even_or_odd = 'even'
    else:
        even_or_odd = 'odd'

    if n < 0:
        sign = 'negative'
    else:
        sign = 'positive'

    print(n, 'is', even_or_odd, 'and', sign)

for n in numbers:
    describe_number(n)

1 is odd and positive
2 is even and positive
3 is odd and positive
4 is even and positive
5 is odd and positive


Running this on my `numbers` list `[1, 2, 3, 4, 5]` shows that 1, 3, and 5 are odd, while 2 and 4 are even, and all five numbers are positive. This matches what I'd expect just from looking at the list.